# DecSelfMask Package Example

This notebook shows the class-first API for the full DecSelfMask workflow: relevance calculation, sequence creation, DecSelfMask training, SFT, and classification head training.

The cells are written as a usage guide. Heavy training calls are included, but left commented so you can run them selectively.

In [ ]:
# If needed, install the package in editable mode before running this notebook.
# !pip install -e .

from dec_self_mask import (
    RelevanceCalculator,
    RelevanceCalculatorConfig,
    DecSelfMaskSequencesMaker,
    SequenceMakerConfig,
    DecSelfMaskTrainingArguments,
    DecSelfMaskTrainer,
    SFTTrainer,
    SFTTrainerConfig,
    ClassificationHeadTrainer,
    ClassificationHeadTrainerConfig,
)

print('Imported DecSelfMask package successfully.')

## 1. Calculate relevance scores

This step scores each note against the target items and writes the relevancy JSON files.

In [ ]:
relevance = RelevanceCalculator(
    RelevanceCalculatorConfig(
        model_name="meta-llama/Llama-3.2-1B-Instruct",
        data_path="wikimedia/wikipedia",
        data_config="20231101.es",
        data_split="train",
        id_column_name="id",
        text_column_name="text",
        start_from_note=0,
        end_at_note=2,
        max_text_length=100,
        file_with_targets="data/targets_for_self_masking.txt",
        path_save="data/",
        cache_dir="/data01/pferrazzi/.cache",
    )
)

# Uncomment to run:
# relevance_output = relevance.calculate()
# relevance_output.keys()

## 2. Build DecSelfMask training sequences

This step reads the relevance output and creates the masked sequence dataset used for DecSelfMask training.

In [ ]:
sequences = DecSelfMaskSequencesMaker(
    SequenceMakerConfig(
        input_path="data/a_attention_relevancy_unannotated/wikipedia/Llama-3.2-1B-Instruct/combined_mid.json",
        model_name="meta-llama/Llama-3.2-1B-Instruct",
        smoothing_method="gaussian",
        threshold_upper=0.4,
        threshold_lower=0.2,
        type_of_masking="all_group",
        output_dir="data/a_train_sequences",
        hf_account_name="ferrazzipietro",
    )
)

# Uncomment to run:
# datasets = sequences.build_datasets()
# sequences.save_local(datasets)
# or: sequences.push_to_hub(datasets)

## 3. Train DecSelfMask

This uses the training config and the sequence dataset prepared above. The class exposes `train()` and `evaluate()`.

You must define a `yaml` file with the training configuration. You can use as example the one at `train_configs/DecSelfMask/llama_1b.yaml`

In [ ]:
dec_self_mask_trainer = DecSelfMaskTrainer(
    DecSelfMaskTrainingArguments(
        custom_config_file="train_configs/DecSelfMask/llama_1b.yaml",
        train_data_path="YOUR_PATH/gaussian_Llama-3.1-8B-Instruct",
        train_data_split="train",
        val_data_path="YOUR_PATH/gaussian_Llama-3.1-8B-Instruct",
        val_data_split="validation",
        cache_dir="/workspace/.cache",
        task_type="dec_self_mask",
    )
)

# Uncomment to run:
# dec_self_mask_trainer.train()
# dec_self_mask_metrics = dec_self_mask_trainer.evaluate()
# dec_self_mask_metrics

## 4. Run SFT

This step fine-tunes the model on the SFT task using Hugging Face training arguments.

In [ ]:
sft = SFTTrainer(
    SFTTrainerConfig(
        custom_config_file="train_configs/sft/llama_1b.yaml",
        train_data_path="ferrazzipietro/crf-second-batch-item-by-item-balanced",
        train_data_split="train",
        val_data_split="validation",
        test_data_split="validation",
        cache_dir="/workspace/.cache",
        task_type="sft_task",
        tag_token_start="<sft_item>",
        item_column_name="sft_item",
        options_column_name="options",
    )
)

# Uncomment to run:
# sft.train()
# sft_metrics = sft.evaluate()
# sft_metrics

## 5. Train the classification head

This final step trains a classifier on top of the DecSelfMask model and can also evaluate the best checkpoint.

In [ ]:
classifier = ClassificationHeadTrainer(
    ClassificationHeadTrainerConfig(
        model_path="ferrazzipietro/DecSelfMask-Llama-3.2-1B-Instruct",
        dataset_path="ferrazzipietro/crf-second-batch-item-by-item-balanced",
        target_col_name="crf_item",
        label_col_name="label",
        freeze_lm=True,
        train_max_size=32,
        val_max_size=32,
        test_max_size=32,
        train_batch_size=64,
        eval_batch_size=64,
        num_epochs=1,
        eval_every_percent=0.5,
        cache_dir="/data01/pferrazzi/.cache",
        type_of_prompt="mask",
        use_non_linearity=True,
        item="all",
    )
)

# Uncomment to run:
# classifier.train()
# classifier_metrics = classifier.evaluate()
# classifier_aggregate = classifier.aggregate_results()
# classifier_metrics, classifier_aggregate

## Next steps

If you want, you can now replace the placeholder paths with your own dataset and config paths, then run each cell in order.